<a href="https://colab.research.google.com/github/marikhakhishvili/Facial-Expression-Recognition-Challenge/blob/main/model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install kaggle wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 14.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
! mkdir ~/.kaggle

In [5]:
!cp /content/drive/MyDrive/cs231n/assignments/assignment4/kaggle.json ~/.kaggle/kaggle.json

In [6]:
! chmod 600 ~/.kaggle/kaggle.json

download competition data

In [7]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:01<00:00, 166MB/s]



In [8]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [9]:
import wandb
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


check if there are any problems with the data


In [9]:
"""
Sanity check for FER2013 pipeline.

1. Builds the smallest possible model: 1 conv layer + FC.
2. Checks initial loss is close to -ln(1/7) ~= 1.9459 (random baseline, 7 classes).
3. Trains on a fixed ~20-image subset and checks loss -> ~0 (model can memorize).
4. Logs everything to Wandb as run_0_sanity_check.

Adjust CSV_PATH / column names to match your actual train.csv (some FER2013
versions have an "emotion"+"pixels" only train.csv, others also have "Usage").
"""

import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb

CSV_PATH = "train.csv"   # update to your file path
NUM_SAMPLES = 20
NUM_CLASSES = 7
NUM_STEPS = 300
LR = 1e-2
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)


class TinyFERSubset(Dataset):
    def __init__(self, csv_path, n_samples=20):
        df = pd.read_csv(csv_path)

        # If a "Usage" column exists, restrict to the training split
        if "Usage" in df.columns:
            df = df[df["Usage"] == "Training"]

        df = df.reset_index(drop=True).iloc[:n_samples]

        images = []
        for pixel_str in df["pixels"]:
            arr = np.array(pixel_str.split(), dtype=np.float32).reshape(48, 48)
            images.append(arr)

        self.images = np.stack(images) / 255.0   # (N, 48, 48), normalized
        self.labels = df["emotion"].values.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx]).unsqueeze(0).float()  # (1, 48, 48)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label


class TinyNet(nn.Module):
    """Smallest reasonable model: 1 conv layer + FC."""

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)  # 48 -> 24
        self.fc = nn.Linear(8 * 24 * 24, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.flatten(1)
        return self.fc(x)


def run_sanity_check():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    wandb.init(
        project="fer2013-assignment",
        group="sanity_checks",
        name="run_0_sanity_check",
        config={
            "model": "TinyNet (1 conv + FC)",
            "num_samples": NUM_SAMPLES,
            "lr": LR,
            "num_steps": NUM_STEPS,
        },
    )

    dataset = TinyFERSubset(CSV_PATH, n_samples=NUM_SAMPLES)
    loader = DataLoader(dataset, batch_size=NUM_SAMPLES, shuffle=True)

    model = TinyNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    images, labels = next(iter(loader))
    images, labels = images.to(device), labels.to(device)

    # --- Check 1: initial loss vs random baseline ---
    model.eval()
    with torch.no_grad():
        initial_loss = criterion(model(images), labels).item()

    expected_random_loss = -math.log(1.0 / NUM_CLASSES)
    print(f"Initial loss: {initial_loss:.4f}  (expected ~{expected_random_loss:.4f})")

    wandb.log({
        "sanity/initial_loss": initial_loss,
        "sanity/expected_random_loss": expected_random_loss,
    }, step=0)

    # --- Check 2: overfit the tiny subset ---
    model.train()
    for step in range(1, NUM_STEPS + 1):
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        acc = (logits.argmax(dim=1) == labels).float().mean().item()
        wandb.log({"sanity/loss": loss.item(), "sanity/accuracy": acc}, step=step)

        if step % 50 == 0 or step == 1:
            print(f"Step {step:4d} | loss {loss.item():.4f} | acc {acc:.2f}")

    final_loss, final_acc = loss.item(), acc
    print("\n--- Summary ---")
    print(f"Initial loss: {initial_loss:.4f} vs expected {expected_random_loss:.4f}")
    print(f"Final loss after {NUM_STEPS} steps: {final_loss:.4f}, accuracy: {final_acc:.2f}")

    if final_loss < 0.05 and final_acc > 0.95:
        print("PASS: model memorized the subset -> forward/backward pass is correct.")
    else:
        print("WARNING: did not fully overfit. Check data loading, labels, LR, or model.")

    wandb.summary.update({
        "initial_loss": initial_loss,
        "expected_random_loss": expected_random_loss,
        "final_loss": final_loss,
        "final_accuracy": final_acc,
    })
    wandb.finish()


if __name__ == "__main__":
    run_sanity_check()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mkhak23 (mkhak23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Initial loss: 2.0955  (expected ~1.9459)
Step    1 | loss 2.0955 | acc 0.00
Step   50 | loss 0.0006 | acc 1.00
Step  100 | loss 0.0001 | acc 1.00
Step  150 | loss 0.0001 | acc 1.00
Step  200 | loss 0.0001 | acc 1.00
Step  250 | loss 0.0001 | acc 1.00
Step  300 | loss 0.0000 | acc 1.00

--- Summary ---
Initial loss: 2.0955 vs expected 1.9459
Final loss after 300 steps: 0.0000, accuracy: 1.00
PASS: model memorized the subset -> forward/backward pass is correct.


sanity/accuracy,▁▁▇█████████████████████████████████████
sanity/expected_random_loss,▁
sanity/initial_loss,▁
sanity/loss,▄▇█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
expected_random_loss,1.94591
final_accuracy,1
final_loss,5e-05
initial_loss,2.0955
sanity/accuracy,1
sanity/expected_random_loss,1.94591
sanity/initial_loss,2.0955


Preprocessing

In [11]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb
df = pd.read_csv('./train.csv')
def pixels_to_image_array(pixels_str):
    """
    Converts a space-separated pixel string into a normalized
    1 x 48 x 48 NumPy array suitable for PyTorch.
    """

    # Convert string to NumPy array of floats
    pixels = np.array(pixels_str.split(), dtype=np.float32)

    # Reshape into a 48x48 image
    image = pixels.reshape(48, 48)

    # Normalize pixel values from [0, 255] to [0, 1]
    image = image / 255.0

    # Add channel dimension: (48, 48) -> (1, 48, 48)
    image = np.expand_dims(image, axis=0)

    return image

# Apply preprocessing to every image
df["pixels"] = df["pixels"].apply(pixels_to_image_array)

# Check the shape of the first image
print(df["pixels"].iloc[0].shape)
# Expected output: (1, 48, 48)

(1, 48, 48)


In [12]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

X = np.stack(df["pixels"].values)   # shape: (N, 1, 48, 48)
y = df["emotion"].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
print("\nEmotion distribution in New Training Set:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nEmotion distribution in Validation Set:")
print(pd.Series(y_val).value_counts(normalize=True))


Emotion distribution in New Training Set:
3    0.251317
6    0.172944
4    0.168241
2    0.142683
0    0.139156
5    0.110463
1    0.015196
Name: proportion, dtype: float64

Emotion distribution in Validation Set:
3    0.251306
6    0.172936
4    0.168234
2    0.142807
0    0.139150
5    0.110414
1    0.015152
Name: proportion, dtype: float64


In [14]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

Model

In [15]:
class CNN_Model2(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [17]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / len(loader), correct / total

Training

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model2().to(device)

criterion = nn.CrossEntropyLoss()

In [17]:
wandb.init(
    project="fer2013-assignment",
    name="model2_deepercnn_lr5e-4_bs64",
    config={
        "model": "CNN_Model2",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 25,
        "optimizer": "Adam",
        "notes": "deeper CNN + batchnorm"
    }
)

config = wandb.config

In [19]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

In [21]:
EPOCHS = 25

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

Epoch 1/25
Train Loss: 1.5393, Train Acc: 0.4029
Val Loss: 1.3318, Val Acc: 0.4871
Epoch 2/25
Train Loss: 1.3011, Train Acc: 0.5029
Val Loss: 1.2723, Val Acc: 0.5077
Epoch 3/25
Train Loss: 1.1885, Train Acc: 0.5503
Val Loss: 1.2635, Val Acc: 0.5225
Epoch 4/25
Train Loss: 1.1085, Train Acc: 0.5810
Val Loss: 1.2323, Val Acc: 0.5273
Epoch 5/25
Train Loss: 1.0178, Train Acc: 0.6180
Val Loss: 1.1667, Val Acc: 0.5681
Epoch 6/25
Train Loss: 0.9402, Train Acc: 0.6447
Val Loss: 1.2046, Val Acc: 0.5505
Epoch 7/25
Train Loss: 0.8584, Train Acc: 0.6832
Val Loss: 1.1440, Val Acc: 0.5831
Epoch 8/25
Train Loss: 0.7629, Train Acc: 0.7187
Val Loss: 1.1788, Val Acc: 0.5724
Epoch 9/25
Train Loss: 0.6719, Train Acc: 0.7541
Val Loss: 1.2509, Val Acc: 0.5719
Epoch 10/25
Train Loss: 0.5765, Train Acc: 0.7930
Val Loss: 1.3206, Val Acc: 0.5782
Epoch 11/25
Train Loss: 0.4709, Train Acc: 0.8377
Val Loss: 1.3134, Val Acc: 0.5758
Epoch 12/25
Train Loss: 0.3942, Train Acc: 0.8618
Val Loss: 1.3884, Val Acc: 0.5742
E

In [1]:
wandb.finish()

NameError: name 'wandb' is not defined

Training with smaller learning rate and less epochs

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model2().to(device)

criterion = nn.CrossEntropyLoss()

In [19]:
wandb.init(
    project="fer2013-assignment",
    name="model2_deepercnn_lr1e-4_bs64",
    config={
        "model": "CNN_Model2",
        "lr": 1e-4,
        "batch_size": 64,
        "epochs": 10,
        "optimizer": "Adam",
        "notes": "deeper CNN + batchnorm"
    }
)

config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mkhak23 (mkhak23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [21]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

In [22]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

Epoch 1/10
Train Loss: 1.5919, Train Acc: 0.3799
Val Loss: 1.4690, Val Acc: 0.4450
Epoch 2/10
Train Loss: 1.3916, Train Acc: 0.4788
Val Loss: 1.3488, Val Acc: 0.4922
Epoch 3/10
Train Loss: 1.2913, Train Acc: 0.5171
Val Loss: 1.3438, Val Acc: 0.4889
Epoch 4/10
Train Loss: 1.2068, Train Acc: 0.5509
Val Loss: 1.2717, Val Acc: 0.5219
Epoch 5/10
Train Loss: 1.1330, Train Acc: 0.5837
Val Loss: 1.2412, Val Acc: 0.5319
Epoch 6/10
Train Loss: 1.0696, Train Acc: 0.6090
Val Loss: 1.2578, Val Acc: 0.5289
Epoch 7/10
Train Loss: 1.0064, Train Acc: 0.6360
Val Loss: 1.2322, Val Acc: 0.5387
Epoch 8/10
Train Loss: 0.9332, Train Acc: 0.6661
Val Loss: 1.2582, Val Acc: 0.5322
Epoch 9/10
Train Loss: 0.8667, Train Acc: 0.6940
Val Loss: 1.2438, Val Acc: 0.5376
Epoch 10/10
Train Loss: 0.8011, Train Acc: 0.7201
Val Loss: 1.3155, Val Acc: 0.5324


In [23]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▄▅▅▆▆▇▇█
train_loss,█▆▅▅▄▃▃▂▂▁
val_acc,▁▅▄▇▇▇████
val_loss,█▄▄▂▁▂▁▂▁▃
epoch,10
train_acc,0.72012
train_loss,0.80111
val_acc,0.53239
val_loss,1.31551


training with lr 5e-4, epochs 10

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model2().to(device)

criterion = nn.CrossEntropyLoss()

In [25]:
wandb.init(
    project="fer2013-assignment",
    name="model2_deepercnn_lr5e-4_bs64_epochs10",
    config={
        "model": "CNN_Model2",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 10,
        "optimizer": "Adam",
        "notes": "deeper CNN + batchnorm"
    }
)

config = wandb.config

In [26]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

In [27]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

Epoch 1/10
Train Loss: 1.5580, Train Acc: 0.3913
Val Loss: 1.3753, Val Acc: 0.4669
Epoch 2/10
Train Loss: 1.3034, Train Acc: 0.5021
Val Loss: 1.2354, Val Acc: 0.5268
Epoch 3/10
Train Loss: 1.1966, Train Acc: 0.5472
Val Loss: 1.2178, Val Acc: 0.5326
Epoch 4/10
Train Loss: 1.1036, Train Acc: 0.5812
Val Loss: 1.2105, Val Acc: 0.5357
Epoch 5/10
Train Loss: 1.0289, Train Acc: 0.6130
Val Loss: 1.2829, Val Acc: 0.5246
Epoch 6/10
Train Loss: 0.9430, Train Acc: 0.6485
Val Loss: 1.1585, Val Acc: 0.5660
Epoch 7/10
Train Loss: 0.8716, Train Acc: 0.6764
Val Loss: 1.1328, Val Acc: 0.5742
Epoch 8/10
Train Loss: 0.7641, Train Acc: 0.7218
Val Loss: 1.1372, Val Acc: 0.5752
Epoch 9/10
Train Loss: 0.6688, Train Acc: 0.7587
Val Loss: 1.1807, Val Acc: 0.5787
Epoch 10/10
Train Loss: 0.5718, Train Acc: 0.7954
Val Loss: 1.2782, Val Acc: 0.5761


In [28]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▄▄▅▅▆▇▇█
train_loss,█▆▅▅▄▄▃▂▂▁
val_acc,▁▅▅▅▅▇████
val_loss,█▄▃▃▅▂▁▁▂▅
epoch,10
train_acc,0.7954
train_loss,0.5718
val_acc,0.57611
val_loss,1.27818
